In [2]:
from pathlib import Path
import pandas as pd

# --------------------------------------------------
# 1. Locate the existing CSV files
# --------------------------------------------------
root = Path.cwd().parent

reference_file = root / "5. Reference Rankings" / "reference_ranking.csv"
tri_file = root / "7. electre_tri_rc_fixed_outputs" / "new_tri_rc_assignments.csv"

output_folder = root / "6. Final Comparison"
output_folder.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 2. Load existing results
# --------------------------------------------------
reference = pd.read_csv(reference_file)
tri = pd.read_csv(tri_file)

print("Reference CSV columns:", reference.columns.tolist())
print("TRI-rC CSV columns:", tri.columns.tolist())

# Check that required columns exist
assert {"REF_AREA", "Rank"}.issubset(reference.columns)
assert {
    "REF_AREA", "lower_class", "upper_class",
    "class_interval", "precise_assignment"
}.issubset(tri.columns)

# --------------------------------------------------
# 3. Convert existing reference ranks into quintiles
# --------------------------------------------------
# Rank 1 = best performance, hence Quintile 5 = best.
# No reference scores or ranks are recalculated.

reference["reference_quintile"] = pd.qcut(
    reference["Rank"],
    q=5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

# --------------------------------------------------
# 4. Join reference quintiles with TRI-rC assignments
# --------------------------------------------------
comparison = reference[
    ["REF_AREA", "REF_AREA_LABEL", "Rank",
     "Reference score", "reference_quintile"]
].merge(
    tri[
        ["REF_AREA", "country", "lower_class", "upper_class",
         "class_interval", "precise_assignment"]
    ],
    on="REF_AREA",
    how="inner",
    validate="one_to_one"
)

assert len(comparison) == 96, (
    f"Expected 96 matched economies, found {len(comparison)}"
)

# --------------------------------------------------
# 5. Identify precise and imprecise assignments
# --------------------------------------------------
# Handles Boolean or text values in the CSV.
if comparison["precise_assignment"].dtype == object:
    precise_flag = (
        comparison["precise_assignment"]
        .astype(str).str.strip().str.lower()
        .map({"true": True, "false": False})
    )
else:
    precise_flag = comparison["precise_assignment"].astype(bool)

if precise_flag.isna().any():
    raise ValueError("Unexpected values in precise_assignment column.")

precise = comparison[precise_flag].copy()

# --------------------------------------------------
# 6. Create the 5 x 5 table
# --------------------------------------------------
table = pd.crosstab(
    precise["reference_quintile"],
    precise["lower_class"]
).reindex(
    index=[1, 2, 3, 4, 5],
    columns=[1, 2, 3, 4, 5],
    fill_value=0
)

table.index.name = "Reference quintile"
table.columns.name = "TRI-rC class"

# --------------------------------------------------
# 7. Calculate agreement statistics
# --------------------------------------------------
same_class = (
    precise["reference_quintile"] == precise["lower_class"]
).sum()

within_interval = (
    (comparison["reference_quintile"] >= comparison["lower_class"])
    & (comparison["reference_quintile"] <= comparison["upper_class"])
).sum()

print("\n--- Comparison results ---")
print("Total economies:", len(comparison))
print("Precise assignments:", len(precise))
print("Imprecise assignments:", len(comparison) - len(precise))
print(f"Same-class agreement: {same_class}/{len(precise)} "
      f"({100 * same_class / len(precise):.1f}%)")
print(f"Quintile within TRI-rC interval: {within_interval}/"
      f"{len(comparison)} "
      f"({100 * within_interval / len(comparison):.1f}%)")

print("\n--- 5 x 5 cross-tabulation ---")
print(table)

# --------------------------------------------------
# 8. Save the outputs
# --------------------------------------------------
comparison.to_csv(
    output_folder / "tri_rc_vs_reference_quintiles.csv",
    index=False
)

table.to_csv(
    output_folder / "tri_rc_precise_cross_tab.csv"
)

print("\nSaved comparison files to:", output_folder)

Reference CSV columns: ['REF_AREA', 'REF_AREA_LABEL', 'Aviation connectivity', 'Aviation dwell time', 'B2B postal connectivity', 'B2B postal delivery time', 'Maritime connectivity', 'Container dwell time', 'Reference score', 'Rank']
TRI-rC CSV columns: ['REF_AREA', 'country', 'lower_class', 'upper_class', 'class_interval', 'precise_assignment', 'bounds_consistent', 'lambda', 'b0_boundary_assumption', 'sigma_alt_Class1', 'sigma_Class1_alt', 'veto_alt_Class1', 'veto_Class1_alt', 'sigma_alt_Class2', 'sigma_Class2_alt', 'veto_alt_Class2', 'veto_Class2_alt', 'sigma_alt_Class3', 'sigma_Class3_alt', 'veto_alt_Class3', 'veto_Class3_alt', 'sigma_alt_Class4', 'sigma_Class4_alt', 'veto_alt_Class4', 'veto_Class4_alt', 'sigma_alt_Class5', 'sigma_Class5_alt', 'veto_alt_Class5', 'veto_Class5_alt', 'lower_rule_trace', 'upper_rule_trace']

--- Comparison results ---
Total economies: 96
Precise assignments: 68
Imprecise assignments: 28
Same-class agreement: 47/68 (69.1%)
Quintile within TRI-rC interval: